# Gradient-boosting player model

- **Question:** Does a nonlinear tree ensemble improve future-season player projections enough to beat Ridge and transparent baselines?
- **Data used:** Canonical cutoff-safe player-season features and targets, persisted Phase 4 model/prediction rows, and the Phase 4 JSON report when present.
- **Unit of observation:** One non-rookie player and prediction season, modeled separately by position and target.
- **Target:** Next-season fantasy points per active game, games active, or total fantasy points.
- **Feature cutoff:** Only information available before the prediction season may enter the feature row.
- **Validation strategy:** Chronological inner tuning, pooled 2020-2024 outer validation for selection, and an untouched 2025 test.
- **Interpretation caveat:** Permutation importance and partial dependence describe fitted associations. Extra flexibility is useful only when it generalizes out of time.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError("Could not find the repository root.")


def table_exists(connection: duckdb.DuckDBPyConnection, table_name: str) -> bool:
    row = connection.execute(
        "SELECT count(*) FROM information_schema.tables WHERE table_name = ?",
        [table_name],
    ).fetchone()
    return bool(row and row[0])


PROJECT_ROOT = find_project_root()
WAREHOUSE_PATH = PROJECT_ROOT / "data" / "warehouse" / "fantasy_football.duckdb"
REPORT_PATH = PROJECT_ROOT / "docs" / "PHASE_4_MODEL_EVALUATION.json"
phase4_report = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.is_file() else {}
print({"warehouse_ready": WAREHOUSE_PATH.is_file(), "report_ready": bool(phase4_report)})

## Inspect the deterministic search space

Phase 4 searches a deliberately compact grid. Internal estimator early stopping is disabled so a random internal validation split cannot replace the project's chronological folds.

In [ ]:
from fantasy_draft_ai.models.player_projection.config import (
    HIST_GRADIENT_BOOSTING,
    PlayerModelConfig,
)
from fantasy_draft_ai.models.player_projection.pipelines import (
    build_pipeline,
    candidate_parameters,
)

config = PlayerModelConfig()
hgb_grid = pd.DataFrame(candidate_parameters(HIST_GRADIENT_BOOSTING, config))
unfitted_pipeline = build_pipeline(HIST_GRADIENT_BOOSTING, config)
print(unfitted_pipeline.named_steps["estimator"])
hgb_grid

In [ ]:
hgb_models = pd.DataFrame()
hgb_predictions = pd.DataFrame()
if WAREHOUSE_PATH.is_file():
    with duckdb.connect(str(WAREHOUSE_PATH), read_only=True) as connection:
        if table_exists(connection, "player_projection_models"):
            hgb_models = connection.execute(
                """
                SELECT run_id, model_id, position, target_name, training_seasons,
                       training_rows, hyperparameters, model_card_path
                FROM player_projection_models
                WHERE model_family = 'hist_gradient_boosting'
                ORDER BY position, target_name
                """
            ).df()
        if table_exists(connection, "player_projection_predictions"):
            hgb_predictions = connection.execute(
                """
                SELECT player_id, prediction_season, position, target_name, fold_label,
                       predicted_value, actual_value, training_max_season
                FROM player_projection_predictions
                WHERE model_family = 'hist_gradient_boosting'
                  AND actual_value IS NOT NULL
                ORDER BY prediction_season, position, target_name, player_id
                """
            ).df()

if hgb_models.empty:
    print("No persisted gradient-boosted models yet. Run Phase 4 training first.")
else:
    display(hgb_models)

## Keep validation and test visibly separate

The table and chart below use only real persisted predictions. Validation summarizes model selection evidence; 2025 is displayed afterward as the frozen test.

In [ ]:
if hgb_predictions.empty:
    hgb_metrics = pd.DataFrame()
    print("No evaluable gradient-boosting predictions are stored yet.")
else:
    assert (hgb_predictions["training_max_season"] < hgb_predictions["prediction_season"]).all()
    hgb_metrics = (
        hgb_predictions.assign(
            period=lambda frame: frame["prediction_season"].map(
                lambda season: "test_2025" if season == 2025 else "validation_2020_2024"
            ),
            absolute_error=lambda frame: (frame["predicted_value"] - frame["actual_value"]).abs(),
        )
        .groupby(["period", "position", "target_name"], dropna=False)
        .agg(rows=("absolute_error", "size"), mae=("absolute_error", "mean"))
        .reset_index()
    )
    display(hgb_metrics)

    chart = hgb_metrics.pivot_table(
        index=["position", "target_name"], columns="period", values="mae"
    )
    chart.plot(kind="bar", figsize=(11, 5), title="Stored HGB MAE: validation versus test")
    plt.ylabel("Mean absolute error")
    plt.tight_layout()
    plt.show()

## Read model-agnostic explanations

Permutation importance asks how much held-out scoring worsens when an input is shuffled. Partial dependence traces the average fitted response as one numeric input changes over its observed range. Neither is a causal experiment.

In [ ]:
global_explanations = phase4_report.get("global_explanations", [])
hgb_explanations = [
    section
    for section in global_explanations
    if section.get("model_family") == "hist_gradient_boosting"
]
importance_records = [
    {
        "position": section.get("position"),
        "target_name": section.get("target_name"),
        **record,
    }
    for section in hgb_explanations
    for record in section.get("importance", [])
]
response_records = [
    {
        "position": section.get("position"),
        "target_name": section.get("target_name"),
        **record,
    }
    for section in hgb_explanations
    for record in section.get("feature_responses", [])
]
if importance_records:
    display(pd.DataFrame(importance_records))
else:
    print("No permutation-importance section is available in the Phase 4 report yet.")

if response_records:
    display(pd.DataFrame(response_records))
else:
    print("No partial-dependence section is available in the Phase 4 report yet.")

## Exercise

Pick one position and target after training. Compare its grid settings, validation MAE, and frozen test MAE. Then identify one likely nonlinear interaction and write an association-safe interpretation of one importance result. Finally, confirm that a learned candidate is promoted only when it has lower validation MAE and the paired-bootstrap learned-minus-baseline 95% interval stays below zero; ties and inconclusive intervals retain the transparent baseline.